In [1]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Load EfficientNet-B3 (Native 300x300 resolution)
base_model = EfficientNetB3(
    weights='imagenet', include_top=False, input_shape=(300, 300, 3))
base_model.trainable = False

# 2. Build the top layers
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(5, activation='softmax')  # Your 5 DR stages
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 3. Setup Data Flow (Point this to your Drive folder)
train_dir = '/content/drive/MyDrive/DRdataset_Processed'  # Double check this path!

# 1. Setup the Generator with a 20% split for validation
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

# 2. This is the TRAINING part (80% of your data)
train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(300, 300),
    batch_size=32,
    class_mode='sparse',
    subset='training'
)

# 3. THIS IS THE MISSING PART: The VALIDATION part (20% of your data)
validation_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(300, 300),
    batch_size=32,
    class_mode='sparse',
    subset='validation'
)

print("Both generators are now defined!")

43941136/43941136 ━━━━━━━━━━━━━━━━━━━━ 318s 7us/step


FileNotFoundError: [WinError 3] The system cannot find the path specified: '/content/drive/MyDrive/DRdataset_Processed'

In [ ]:
# This starts the actual 'learning' process
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator,  # This checks accuracy on unseen images
    verbose=1
)

In [ ]:
# 1. Unfreeze the base model
base_model.trainable = True

# 2. Re-compile with a VERY SMALL learning rate
# (We use 1e-5 so we don't destroy the pre-trained knowledge)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 3. Train for another 10-20 epochs
fine_tune_history = model.fit(
    train_generator,
    epochs=20,
    validation_data=validation_generator
)

In [ ]:
model.save('/content/drive/MyDrive/DR_FineTuned_B3_v1.keras')

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Get the ground truth labels and the model's predictions
# We reset the generator to ensure it starts from the first image
validation_generator.reset()
y_true = validation_generator.classes
preds = model.predict(validation_generator)
y_pred = np.argmax(preds, axis=1)

# 2. Create the matrix
cm = confusion_matrix(y_true, y_pred)

In [ ]:
# 1. Reset generator and get predictions
validation_generator.reset()
# It is safer to predict on the full generator to get the right order
preds = model.predict(validation_generator)
y_pred = np.argmax(preds, axis=1)
y_true = validation_generator.classes

# 2. Get class names from the generator itself to be 100% accurate
class_labels = list(validation_generator.class_indices.keys())

# 3. Create the matrix
cm = confusion_matrix(y_true, y_pred)

# 4. Plot
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('Confusion Matrix: Diabetic Retinopathy')
plt.ylabel('Actual Stage')
plt.xlabel('Predicted Stage')
plt.show()

# 5. FIXED LINE: Changed 'target_size' to 'target_names'
print(classification_report(y_true, y_pred, target_names=class_labels))

In [ ]:
# Save using the modern format
model.save('/content/drive/MyDrive/DR_EfficientNetB3_Model.keras')
print("Model saved to Drive in the modern .keras format!")

In [ ]:
import os
path = '/content/drive/MyDrive/DRdataset_Processed'
if os.path.exists(path):
    print(f"Success! Found {len(os.listdir(path))} subfolders.")
else:
    print("Path still not found. Check the sidebar again!")